# Library Imports and Setup

In [1]:
import sqlite3
import os
import pandas as pd
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

# Load environment variables from .env file
load_dotenv('../.env')

token = os.getenv("GROQ_API_KEY") # Or OPENAI_API_KEY depending on your setup
endpoint = os.getenv("BASE_URL")
model_name = os.getenv("MODEL_NAME")

if not token:
    raise ValueError("API KEY environment variable not set.")

# LLM Model setup
llm = ChatOpenAI(
    model_name=model_name,
    openai_api_key=token,
    openai_api_base=endpoint,
    temperature=0.5,
)

# Convert CSV to SQLite

In [2]:
# Folder paths
db_folder = '../sql/'
csv_folder = '../csvfile/' # Make sure your downloaded CSV files are in this folder

os.makedirs(db_folder, exist_ok=True)

print("Reading CSVs and creating SQLite databases...")

# 1. Institutions Database
df_inst = pd.read_csv(f'{csv_folder}institution.csv') 
conn_inst = sqlite3.connect(f'{db_folder}institutions.db')
df_inst.to_sql('institutions', conn_inst, if_exists='replace', index=False)
conn_inst.close()

# 2. Hospitals Database
df_hosp = pd.read_csv(f'{csv_folder}bangladesh_hospitals.csv')
conn_hosp = sqlite3.connect(f'{db_folder}hospitals.db')
df_hosp.to_sql('hospitals', conn_hosp, if_exists='replace', index=False)
conn_hosp.close()

# 3. Restaurants Database
df_rest = pd.read_csv(f'{csv_folder}restaurants.csv') 
conn_rest = sqlite3.connect(f'{db_folder}restaurants.db')
df_rest.to_sql('restaurants', conn_rest, if_exists='replace', index=False)
conn_rest.close()

print("Successfully created all three databases from local CSV files!")

Reading CSVs and creating SQLite databases...
Successfully created all three databases from local CSV files!


# Building the Tools

In [3]:
# --- Web Search Tool ---
WebSearchTool = TavilySearchResults(
    max_results=3,
    name="WebSearchTool",
    description="Use this tool ONLY for general knowledge, definitions, policies, or cultural context about Bangladesh. DO NOT use this for database statistics."
)

# --- Helper Function for Schema ---
def get_schema_and_sample(db_path: str) -> str:
    """Helper function to return the schema and 3 sample rows for a given database."""
    con = sqlite3.connect(db_path)
    try:
        cursor = con.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = [row[0] for row in cursor.fetchall() if not row[0].startswith("sqlite_")]
        
        results = []
        for table in tables:
            cursor.execute(f"SELECT sql FROM sqlite_master WHERE type='table' AND name='{table}';")
            schema_row = cursor.fetchone()
            if schema_row:
                results.append(schema_row[0])
                try:
                    cursor.execute(f'SELECT * FROM "{table}" LIMIT 3;')
                    rows = cursor.fetchall()
                    if rows:
                        col_names = [description[0] for description in cursor.description]
                        sample_data = f"/*\n3 sample rows from {table}:\n" + "\t".join(col_names) + "\n" + "\n".join("\t".join(str(x) for x in row) for row in rows) + "\n*/"
                        results.append(sample_data)
                except Exception as e:
                    results.append(f"Error fetching sample rows: {e}")
        return "\n\n".join(results)
    finally:
        con.close()

# --- Database Tools ---
@tool
def InstitutionsDBTool(command: str) -> str:
    """
    Tool for Institutional Information database of Bangladesh.
    INSTRUCTIONS:
    - To see the database structure, pass the exact word: SCHEMA
    - To execute a query, pass a valid SQL query.
    """
    db_path = "../sql/institutions.db"
    command = command.strip()
    if command.upper() == "SCHEMA":
        return get_schema_and_sample(db_path)
    
    con = sqlite3.connect(db_path)
    try:
        cursor = con.cursor()
        cursor.execute(command)
        res = cursor.fetchall()
        return str(res)
    except Exception as e:
        return f"SQL Error: {e}. Please check the schema and fix your SQL query."
    finally:
        con.close()

@tool
def HospitalsDBTool(command: str) -> str:
    """
    Tool for Bangladeshi Hospitals database.
    INSTRUCTIONS:
    - To see the database structure, pass the exact word: SCHEMA
    - To execute a query, pass a valid SQL query.
    """
    db_path = "../sql/hospitals.db"
    command = command.strip()
    if command.upper() == "SCHEMA":
        return get_schema_and_sample(db_path)
    
    con = sqlite3.connect(db_path)
    try:
        cursor = con.cursor()
        cursor.execute(command)
        res = cursor.fetchall()
        return str(res)
    except Exception as e:
        return f"SQL Error: {e}. Please check the schema and fix your SQL query."
    finally:
        con.close()

@tool
def RestaurantsDBTool(command: str) -> str:
    """
    Tool for Bangladeshi Restaurant Data.
    INSTRUCTIONS:
    - To see the database structure, pass the exact word: SCHEMA
    - To execute a query, pass a valid SQL query.
    """
    db_path = "../sql/restaurants.db"
    command = command.strip()
    if command.upper() == "SCHEMA":
        return get_schema_and_sample(db_path)
    
    con = sqlite3.connect(db_path)
    try:
        cursor = con.cursor()
        cursor.execute(command)
        res = cursor.fetchall()
        return str(res)
    except Exception as e:
        return f"SQL Error: {e}. Please check the schema and fix your SQL query."
    finally:
        con.close()

# Combine all tools
tools = [InstitutionsDBTool, HospitalsDBTool, RestaurantsDBTool, WebSearchTool]

# Main Agent Logic & Testing

In [4]:
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_message = """You are an expert AI Agent managing three databases about Bangladesh and a web search tool.
Your workflow for DATABASE questions MUST be:
1. First, pass the exact word "SCHEMA" to the appropriate database tool to understand the tables and columns.
2. Double-check your SQL query logic based on the returned schema.
3. Pass the final SQL query to the exact same tool to get the data.
4. If you get a SQL error, analyze the error, rewrite the query, and try again.

Your tool routing rules:
- Queries about universities, colleges, or government institutions -> InstitutionsDBTool
- Queries about hospitals, doctors, beds, or clinics -> HospitalsDBTool
- Queries about restaurants, food, or dining -> RestaurantsDBTool
- General knowledge, definitions, or healthcare/food policies in Bangladesh -> WebSearchTool (Do NOT use schema for web search)

Important: After getting the raw data from the database tools, you MUST return the final answer in a friendly, natural human language."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Create the Agent
agent = create_openai_tools_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=False, handle_parsing_errors=True)


In [6]:
# Testing
print("\n--- Example 1: Web Search Query ---")
response1 = agent_executor.invoke({"input": "What is the healthcare policy of Bangladesh?"})
print("\nFinal Answer:\n", response1["output"])



--- Example 1: Web Search Query ---

Final Answer:
 **Bangladesh’s health‑care policy – in a nutshell**

Bangladesh’s most recent national health framework is the **National Health Policy (NHP) 2011**, which builds on earlier policy drafts and the country’s “Vision 2021” development plan. The policy is guided by the constitution’s guarantee that every citizen has a basic right to adequate health care (Article 15 (Ka)) and aims to move the nation toward **universal health coverage** by strengthening both public and private health services.

---

### Core Objectives  

| Goal | What it means |
|------|----------------|
| **Universal access** | Ensure that all people—rural or urban, rich or poor—can obtain essential health services without financial hardship. |
| **Reduce mortality** | Cut child mortality (under‑5) to ≤ 15 per 1,000 live births and maternal mortality to ≤ 1.5 per 1,000 live births (aligned with the Millennium Development Goals and later Sustainable Development Goals). |


In [5]:
# Testing
print("--- Example 2: Database Query ---")
response2 = agent_executor.invoke({"input": "How many government institutions are in Rajshahi?"})
print("\nFinal Answer:\n", response2["output"])

--- Example 2: Database Query ---

Final Answer:
 There are **110 government‑run institutions** located in the Rajshahi division.


In [7]:
# Testing
print("--- Example 3: Database Query ---")
response3 = agent_executor.invoke({"input": "Find restaurants in Chattogram serving biryani."})
print("\nFinal Answer:\n", response3["output"])


--- Example 3: Database Query ---

Final Answer:
 Here are the restaurants in Chattogram that serve biryani:

| Restaurant | Address | Rating | Number of Reviews |
|------------|---------|--------|--------------------|
| Shah Amanat Biryani House | 6QPV+67, Chattogram, Bangladesh | 0.0 (no rating yet) | — (no reviews) |

This is the only biryani‑focused restaurant currently listed in the database for Chattogram. If you need more options or details (like contact info or opening hours), let me know and I can look up additional sources!
